# Train Empathy Model bằng XLM-R từ dữ liệu Empathy public

Notebook này dùng dữ liệu trong:

```text
data/raw/auxiliary_pretrain_baseline/02_empathy/
```

Cách tạo nhãn baseline/pseudo-label:

```text
Response gốc từ EmpatheticDialogues / ESConv -> high_empathy
Response cộc lốc tự tạo -> low_empathy
Response lịch sự nhưng chung chung tự tạo -> medium_empathy
```

Sau này nên fine-tune thêm bằng dữ liệu CSKH tiếng Việt được gán nhãn thật.

In [1]:
!pip install pandas scikit-learn  transformers datasets accelerate tqdm -q

In [2]:
import re
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

AUXILIARY_DATA_DIR = Path("data/raw/auxiliary_pretrain_baseline/02_empathy")
LABELED_DATA_DIRS = [Path("data/processed/empathy"), Path("data/raw/custom_labeled/empathy")]

OUTPUT_DIR = Path("models/empathy_xlm_roberta")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "xlm-roberta-base"
MAX_LENGTH = 256
TEST_SIZE = 0.15
VALID_SIZE = 0.15
RANDOM_SEED = 42
NUM_EPOCHS = 3
BATCH_SIZE = 8
LEARNING_RATE = 2e-5

id2label = {0: "low_empathy", 1: "medium_empathy", 2: "high_empathy"}
label2id = {"low_empathy": 0, "medium_empathy": 1, "high_empathy": 2}

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

print("AUXILIARY_DATA_DIR:", AUXILIARY_DATA_DIR)
print("MODEL_NAME:", MODEL_NAME)
print("OUTPUT_DIR:", OUTPUT_DIR)

d:\learning\CareScore-AI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Torch: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 4050 Laptop GPU
AUXILIARY_DATA_DIR: data\raw\auxiliary_pretrain_baseline\02_empathy
MODEL_NAME: xlm-roberta-base
OUTPUT_DIR: models\empathy_xlm_roberta


## Hàm tiện ích

In [3]:
def normalize_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " <url> ", text)
    text = re.sub(r"\S+@\S+", " <email> ", text)
    text = re.sub(r"\b\d{9,11}\b", " <phone> ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def read_csv_robust(csv_path):
    try:
        return pd.read_csv(csv_path)
    except Exception:
        return pd.read_csv(csv_path, engine="python", on_bad_lines="skip", encoding="utf-8")


def map_empathy_label(value):
    if pd.isna(value):
        return None
    raw = str(value).strip().lower()

    low_labels = ["no_empathy", "none", "low", "low_empathy", "không đồng cảm", "khong dong cam", "ít đồng cảm", "it dong cam"]
    mid_labels = ["medium", "medium_empathy", "neutral", "có đồng cảm", "co dong cam", "trung bình", "trung binh"]
    high_labels = ["high", "high_empathy", "very_high", "strong_empathy", "đồng cảm cao", "dong cam cao", "rất đồng cảm", "rat dong cam"]

    if raw in low_labels:
        return 0
    if raw in mid_labels:
        return 1
    if raw in high_labels:
        return 2

    try:
        score = float(raw)
        if score <= 2:
            return 0
        if score == 3:
            return 1
        if score >= 4:
            return 2
    except Exception:
        return None
    return None


LABEL_COL_CANDIDATES = ["empathy_label", "empathy_score", "label", "score", "rating", "empathy"]

def find_label_column(columns):
    columns_lower = {c.lower(): c for c in columns}
    for cand in LABEL_COL_CANDIDATES:
        if cand.lower() in columns_lower:
            return columns_lower[cand.lower()]
    return None


def build_empathy_input(row):
    conversation_cols = ["conversation_text", "conversation", "dialogue", "dialog", "full_conversation", "text"]
    for col in conversation_cols:
        if col in row and pd.notna(row[col]) and str(row[col]).strip():
            return str(row[col])

    customer_cols = ["customer_text", "user_text", "client_text", "customer", "khach_hang", "prompt", "situation", "context"]
    agent_cols = ["agent_text", "staff_text", "employee_text", "bot_text", "assistant_text", "nhan_vien", "response", "utterance", "output"]

    customer_parts, agent_parts = [], []
    for col in customer_cols:
        if col in row and pd.notna(row[col]) and str(row[col]).strip():
            customer_parts.append(str(row[col]))
    for col in agent_cols:
        if col in row and pd.notna(row[col]) and str(row[col]).strip():
            agent_parts.append(str(row[col]))

    if customer_parts or agent_parts:
        text = ""
        if customer_parts:
            text += "Khách hàng: " + " ".join(customer_parts) + " "
        if agent_parts:
            text += "Nhân viên: " + " ".join(agent_parts)
        return text.strip()

    for col in ["comment", "sentence", "content", "utterance", "response", "input", "output"]:
        if col in row and pd.notna(row[col]) and str(row[col]).strip():
            return str(row[col])
    return ""

## Tạo template gán nhãn thật nếu cần

In [4]:
def create_empathy_annotation_template():
    out_dir = Path("data/processed/empathy")
    out_dir.mkdir(parents=True, exist_ok=True)
    template_path = out_dir / "empathy_annotation_template.csv"

    if not template_path.exists():
        columns = ["conversation_id", "conversation_text", "customer_text", "agent_text", "empathy_label", "empathy_score", "annotator", "notes"]
        pd.DataFrame(columns=columns).to_csv(template_path, index=False, encoding="utf-8-sig")

    guide_path = out_dir / "EMPATHY_LABELING_GUIDE.md"
    guide = """
# Hướng dẫn gán nhãn Empathy

low_empathy: phản hồi cộc lốc/máy móc, score 1-2.
medium_empathy: có xin lỗi/hỗ trợ nhưng còn chung chung, score 3.
high_empathy: thể hiện rõ thấu hiểu, xin lỗi phù hợp, hướng hỗ trợ cụ thể, score 4-5.
"""
    guide_path.write_text(guide.strip(), encoding="utf-8")
    print("Template:", template_path)
    print("Guide:", guide_path)

create_empathy_annotation_template()

Template: data\processed\empathy\empathy_annotation_template.csv
Guide: data\processed\empathy\EMPATHY_LABELING_GUIDE.md


## Load dữ liệu public và tạo pseudo-label

In [5]:
def extract_public_empathy_fields(row):
    customer_text_parts = []
    response = ""

    context_cols = ["prompt", "situation", "context", "emotion", "label", "problem_type", "scenario", "input"]
    for col in context_cols:
        if col in row and pd.notna(row[col]) and str(row[col]).strip():
            customer_text_parts.append(str(row[col]))

    response_cols = ["utterance", "response", "text", "output", "assistant_text", "reply"]
    for col in response_cols:
        if col in row and pd.notna(row[col]) and str(row[col]).strip():
            response = str(row[col])
            break

    customer_text = " ".join(customer_text_parts) if customer_text_parts else "Khách hàng đang chia sẻ một vấn đề hoặc cảm xúc tiêu cực."
    return customer_text, response


def build_auxiliary_empathy_from_public_datasets(aux_dir: Path, max_rows_per_file=None):
    rows = []

    if not aux_dir.exists():
        print("Không tìm thấy folder auxiliary:", aux_dir)
        return pd.DataFrame(rows)

    csv_files = list(aux_dir.rglob("*.csv"))
    print(f"Tìm thấy {len(csv_files)} file CSV trong {aux_dir}")

    low_responses = [
        "Gửi thông tin để kiểm tra.",
        "Bạn cung cấp mã đơn hàng.",
        "Vấn đề này cần chờ xử lý.",
        "Bạn kiểm tra lại giúp.",
        "Bên mình chưa có thông tin.",
        "Không rõ, bạn gửi lại thông tin.",
        "Bạn tự kiểm tra lại giúp.",
    ]

    medium_responses = [
        "Dạ bên em xin lỗi, anh/chị gửi thêm thông tin để em kiểm tra ạ.",
        "Dạ em xin lỗi về bất tiện này, em sẽ kiểm tra lại cho mình ạ.",
        "Dạ em ghi nhận vấn đề của anh/chị và sẽ chuyển bộ phận liên quan kiểm tra ạ.",
        "Dạ em rất tiếc về trường hợp này, anh/chị cho em xin thêm thông tin ạ.",
        "Dạ em xin lỗi, bên em sẽ hỗ trợ kiểm tra trong thời gian sớm nhất ạ.",
        "Dạ em hiểu vấn đề của mình, anh/chị gửi thêm thông tin để em hỗ trợ ạ.",
    ]

    for csv_path in csv_files:
        name_lower = csv_path.name.lower()
        if any(x in name_lower for x in ["prediction", "summary", "dataset_info"]):
            continue

        try:
            df = read_csv_robust(csv_path)
        except Exception as e:
            print(f"Không đọc được: {csv_path} | lỗi: {e}")
            continue

        if max_rows_per_file is not None and len(df) > max_rows_per_file:
            df = df.sample(max_rows_per_file, random_state=RANDOM_SEED)

        print("\\nĐang xử lý:", csv_path)
        print("Columns:", list(df.columns))
        print("Rows:", len(df))

        used_rows = 0
        for _, row in df.iterrows():
            customer_text, original_response = extract_public_empathy_fields(row)
            if not original_response:
                continue

            high_text = "Khách hàng: " + customer_text + " Nhân viên: " + original_response
            rows.append({"text": normalize_text(high_text), "label": 2, "source_file": str(csv_path), "pseudo_label": "high_empathy_from_public_empathy_dataset"})

            low_text = "Khách hàng: " + customer_text + " Nhân viên: " + random.choice(low_responses)
            rows.append({"text": normalize_text(low_text), "label": 0, "source_file": str(csv_path), "pseudo_label": "low_empathy_synthetic"})

            medium_text = "Khách hàng: " + customer_text + " Nhân viên: " + random.choice(medium_responses)
            rows.append({"text": normalize_text(medium_text), "label": 1, "source_file": str(csv_path), "pseudo_label": "medium_empathy_synthetic"})

            used_rows += 1

        print(f"Đã tạo {used_rows * 3} mẫu pseudo-label từ file này.")

    data = pd.DataFrame(rows)
    if not data.empty:
        data = data.drop_duplicates(subset=["text", "label"]).reset_index(drop=True)
    return data


def load_human_labeled_empathy_data(search_dirs):
    rows = []
    for data_dir in search_dirs:
        if not data_dir.exists():
            continue
        for csv_path in data_dir.rglob("*.csv"):
            name_lower = csv_path.name.lower()
            if any(x in name_lower for x in ["prediction", "summary", "dataset_info"]):
                continue
            try:
                df = read_csv_robust(csv_path)
            except Exception as e:
                print(f"Không đọc được: {csv_path} | lỗi: {e}")
                continue

            label_col = find_label_column(df.columns)
            if label_col is None:
                continue

            used_rows = 0
            for _, row in df.iterrows():
                text = build_empathy_input(row)
                label = map_empathy_label(row[label_col])
                if label is None:
                    continue
                text = normalize_text(text)
                if len(text) < 3:
                    continue
                rows.append({"text": text, "label": label, "source_file": str(csv_path), "pseudo_label": "human_labeled"})
                used_rows += 1

            if used_rows > 0:
                print(f"Đã load {used_rows} dòng có nhãn thật từ: {csv_path}")
                print(f"  Label column: {label_col}")

    data = pd.DataFrame(rows)
    if not data.empty:
        data = data.drop_duplicates(subset=["text", "label"]).reset_index(drop=True)
    return data

## Gộp dữ liệu train

In [6]:
df_labeled = load_human_labeled_empathy_data(LABELED_DATA_DIRS)

# Nếu dữ liệu quá nhiều, đổi max_rows_per_file=5000 để chạy thử nhanh.
df_aux = build_auxiliary_empathy_from_public_datasets(AUXILIARY_DATA_DIR, max_rows_per_file=None)

print("\\nDữ liệu empathy có nhãn thật:", len(df_labeled))
print("Dữ liệu auxiliary/pseudo-label:", len(df_aux))

df_list = []
if not df_labeled.empty:
    df_list.append(df_labeled)
if not df_aux.empty:
    df_list.append(df_aux)

if not df_list:
    raise ValueError(
        "Không tìm thấy dữ liệu empathy. Hãy kiểm tra folder "
        "data/raw/auxiliary_pretrain_baseline/02_empathy hoặc thêm dữ liệu gán nhãn vào data/processed/empathy."
    )

df = pd.concat(df_list, ignore_index=True)
df = df.drop_duplicates(subset=["text", "label"]).reset_index(drop=True)

print("\\nTổng số dòng dùng train:", len(df))
print("\\nPhân bố nhãn:")
print(df["label"].map(id2label).value_counts())
print("\\nPhân bố nguồn pseudo-label:")
print(df["pseudo_label"].value_counts())
display(df.head(10))

Tìm thấy 12 file CSV trong data\raw\auxiliary_pretrain_baseline\02_empathy
\nĐang xử lý: data\raw\auxiliary_pretrain_baseline\02_empathy\empathetic_dialogues_contexts\test.csv
Columns: ['Unnamed: 0', 'situation', 'emotion']
Rows: 2542
Đã tạo 0 mẫu pseudo-label từ file này.
\nĐang xử lý: data\raw\auxiliary_pretrain_baseline\02_empathy\empathetic_dialogues_contexts\train.csv
Columns: ['Unnamed: 0', 'situation', 'emotion']
Rows: 19209
Đã tạo 0 mẫu pseudo-label từ file này.
\nĐang xử lý: data\raw\auxiliary_pretrain_baseline\02_empathy\empathetic_dialogues_contexts\validation.csv
Columns: ['Unnamed: 0', 'situation', 'emotion']
Rows: 2756
Đã tạo 0 mẫu pseudo-label từ file này.
\nĐang xử lý: data\raw\auxiliary_pretrain_baseline\02_empathy\empathetic_dialogues_direct\test.csv
Columns: ['conv_id', 'utterance_idx', 'context', 'prompt', 'speaker_idx', 'utterance', 'selfeval', 'tags']
Rows: 5695
Đã tạo 17085 mẫu pseudo-label từ file này.
\nĐang xử lý: data\raw\auxiliary_pretrain_baseline\02_empath

,text,label,source_file,pseudo_label
0,khách hàng: i felt guilty when i was driving h...,2,data\raw\auxiliary_pretrain_baseline\02_empath...,high_empathy_from_public_empathy_dataset
1,khách hàng: i felt guilty when i was driving h...,0,data\raw\auxiliary_pretrain_baseline\02_empath...,low_empathy_synthetic
2,khách hàng: i felt guilty when i was driving h...,1,data\raw\auxiliary_pretrain_baseline\02_empath...,medium_empathy_synthetic
3,khách hàng: i felt guilty when i was driving h...,2,data\raw\auxiliary_pretrain_baseline\02_empath...,high_empathy_from_public_empathy_dataset
4,khách hàng: i felt guilty when i was driving h...,0,data\raw\auxiliary_pretrain_baseline\02_empath...,low_empathy_synthetic
5,khách hàng: i felt guilty when i was driving h...,1,data\raw\auxiliary_pretrain_baseline\02_empath...,medium_empathy_synthetic
6,khách hàng: i felt guilty when i was driving h...,2,data\raw\auxiliary_pretrain_baseline\02_empath...,high_empathy_from_public_empathy_dataset
7,khách hàng: i felt guilty when i was driving h...,0,data\raw\auxiliary_pretrain_baseline\02_empath...,low_empathy_synthetic
8,khách hàng: i felt guilty when i was driving h...,1,data\raw\auxiliary_pretrain_baseline\02_empath...,medium_empathy_synthetic
9,khách hàng: my mother stopped by my house one ...,2,data\raw\auxiliary_pretrain_baseline\02_empath...,high_empathy_from_public_empathy_dataset


## Chia train / validation / test

In [7]:
def split_data(df):
    if len(df) < 30:
        print("CẢNH BÁO: Dữ liệu rất ít. Nên thêm dữ liệu trước khi train thật.")

    label_counts = df["label"].value_counts()
    can_stratify = label_counts.min() >= 2 and len(label_counts) >= 2
    stratify_col = df["label"] if can_stratify else None

    train_df, temp_df = train_test_split(
        df,
        test_size=TEST_SIZE + VALID_SIZE,
        random_state=RANDOM_SEED,
        stratify=stratify_col,
    )

    relative_test_size = TEST_SIZE / (TEST_SIZE + VALID_SIZE)

    if can_stratify and temp_df["label"].value_counts().min() >= 2:
        stratify_temp = temp_df["label"]
    else:
        stratify_temp = None

    valid_df, test_df = train_test_split(
        temp_df,
        test_size=relative_test_size,
        random_state=RANDOM_SEED,
        stratify=stratify_temp,
    )

    print("Train:", len(train_df))
    print("Valid:", len(valid_df))
    print("Test :", len(test_df))

    return train_df.reset_index(drop=True), valid_df.reset_index(drop=True), test_df.reset_index(drop=True)

train_df, valid_df, test_df = split_data(df)
display(train_df.head())

Train: 208814
Valid: 44746
Test : 44747


,text,label,source_file,pseudo_label
0,khách hàng: i just applied for a new job! it i...,0,data\raw\auxiliary_pretrain_baseline\02_empath...,low_empathy_synthetic
1,khách hàng: when i was little_comma_ i believe...,2,data\raw\auxiliary_pretrain_baseline\02_empath...,high_empathy_from_public_empathy_dataset
2,khách hàng: i was afraid when my dad used to s...,1,data\raw\auxiliary_pretrain_baseline\02_empath...,medium_empathy_synthetic
3,khách hàng: even though i've had lots of datin...,1,data\raw\auxiliary_pretrain_baseline\02_empath...,medium_empathy_synthetic
4,khách hàng: i bet i have the best looking apar...,2,data\raw\auxiliary_pretrain_baseline\02_empath...,high_empathy_from_public_empathy_dataset


## Tokenize

In [8]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_dataset = Dataset.from_pandas(train_df[["text", "label"]])
valid_dataset = Dataset.from_pandas(valid_df[["text", "label"]])
test_dataset = Dataset.from_pandas(test_df[["text", "label"]])

def tokenize_function(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)

train_dataset = train_dataset.map(tokenize_function, batched=True)
valid_dataset = valid_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

train_dataset = train_dataset.remove_columns(["text"])
valid_dataset = valid_dataset.remove_columns(["text"])
test_dataset = test_dataset.remove_columns(["text"])

train_dataset.set_format("torch")
valid_dataset.set_format("torch")
test_dataset.set_format("torch")

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(train_dataset)
print(valid_dataset)
print(test_dataset)

d:\learning\CareScore-AI\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Admin\.cache\huggingface\hub\models--xlm-roberta-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Map: 100%|██████████| 44747/44747 [00:01<00:00, 32352.86 examples/s]

Dataset({
    features: ['label', 'input_ids', 'attention_mask'],
    num_rows: 208814
})
Dataset({
    features: ['label', 'input_ids', 'attention_mask'],
    num_rows: 44746
})
Dataset({
    features: ['label', 'input_ids', 'attention_mask'],
    num_rows: 44747
})


## Build model

In [9]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=id2label,
    label2id=label2id,
)

print(model.config)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 6271.56it/s]
[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


XLMRobertaConfig {
  "add_cross_attention": false,
  "architectures": [
    "XLMRobertaForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "dtype": "float32",
  "eos_token_id": 2,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "id2label": {
    "0": "low_empathy",
    "1": "medium_empathy",
    "2": "high_empathy"
  },
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": false,
  "label2id": {
    "high_empathy": 2,
    "low_empathy": 0,
    "medium_empathy": 1
  },
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 514,
  "model_type": "xlm-roberta",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "output_past": true,
  "pad_token_id": 1,
  "position_embedding_type": "absolute",
  "tie_word_embeddings": true,
  "transformers_version": "5.9.0",
  "type_vocab_size": 1,
  "use_cache": true,
  "vocab_size": 250002
}



## Train model

In [10]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        average="weighted",
        zero_division=0,
    )

    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "precision_weighted": precision,
        "recall_weighted": recall,
        "f1_weighted": f1,
    }

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

model.to(device)

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_weighted",
    greater_is_better=True,
    logging_dir=str(OUTPUT_DIR / "logs"),
    logging_steps=50,
    save_total_limit=2,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

Using device: cuda
GPU: NVIDIA GeForce RTX 4050 Laptop GPU


[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Epoch,Training Loss,Validation Loss,Accuracy,Precision Weighted,Recall Weighted,F1 Weighted
1,0.000000,0.000000,1.000000,1.000000,1.000000,1.000000
2,0.000000,0.000000,1.000000,1.000000,1.000000,1.000000
3,0.000000,0.000270,0.999978,0.999978,0.999978,0.999978


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.03s/it]


TrainOutput(global_step=78306, training_loss=0.001312922803429307, metrics={'train_runtime': 113042.7826, 'train_samples_per_second': 5.542, 'train_steps_per_second': 0.693, 'total_flos': 2.6858466597016404e+16, 'train_loss': 0.001312922803429307, 'epoch': 3.0})

## Test model và lưu model

In [11]:
print("Đánh giá trên tập test:")
test_result = trainer.evaluate(test_dataset)
print(test_result)

predictions = trainer.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=-1)
labels = predictions.label_ids

print("\\nClassification report:")
print(
    classification_report(
        labels,
        preds,
        target_names=[id2label[i] for i in range(3)],
        zero_division=0,
    )
)

output_test_path = OUTPUT_DIR / "test_predictions.csv"
result_df = test_df.copy()
result_df["pred_label_id"] = preds
result_df["pred_label"] = [id2label[int(p)] for p in preds]
result_df["true_label"] = [id2label[int(y)] for y in labels]
result_df.to_csv(output_test_path, index=False, encoding="utf-8-sig")

final_dir = OUTPUT_DIR / "final_model"
final_dir.mkdir(parents=True, exist_ok=True)

trainer.save_model(str(final_dir))
tokenizer.save_pretrained(str(final_dir))

with open(final_dir / "label_mapping.json", "w", encoding="utf-8") as f:
    json.dump({"id2label": id2label, "label2id": label2id}, f, ensure_ascii=False, indent=2)

print("Đã lưu prediction:", output_test_path)
print("Đã lưu model:", final_dir)

display(result_df.head(20))

Đánh giá trên tập test:


Training Loss,Validation Loss,Epoch,Accuracy,Precision Weighted,Recall Weighted,F1 Weighted
0.000000,0.000000,3,1.000000,1.000000,1.000000,1.000000


{'eval_loss': 0.0, 'eval_accuracy': 1.0, 'eval_precision_weighted': 1.0, 'eval_recall_weighted': 1.0, 'eval_f1_weighted': 1.0}


\nClassification report:
                precision    recall  f1-score   support

   low_empathy       1.00      1.00      1.00     16227
medium_empathy       1.00      1.00      1.00     15084
  high_empathy       1.00      1.00      1.00     13436

      accuracy                           1.00     44747
     macro avg       1.00      1.00      1.00     44747
  weighted avg       1.00      1.00      1.00     44747



Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.52s/it]

Đã lưu prediction: models\empathy_xlm_roberta\test_predictions.csv
Đã lưu model: models\empathy_xlm_roberta\final_model


,text,label,source_file,pseudo_label,pred_label_id,pred_label,true_label
0,khách hàng: i finally made reservations at the...,2,data\raw\auxiliary_pretrain_baseline\02_empath...,high_empathy_from_public_empathy_dataset,2,high_empathy,high_empathy
1,khách hàng: my eldest niece graduated high sch...,2,data\raw\auxiliary_pretrain_baseline\02_empath...,high_empathy_from_public_empathy_dataset,2,high_empathy,high_empathy
2,khách hàng: once i met someone who told me the...,0,data\raw\auxiliary_pretrain_baseline\02_empath...,low_empathy_synthetic,0,low_empathy,low_empathy
3,khách hàng: my cat pooped on the rug today. i ...,2,data\raw\auxiliary_pretrain_baseline\02_empath...,high_empathy_from_public_empathy_dataset,2,high_empathy,high_empathy
4,khách hàng: some one ran into my car and drove...,1,data\raw\auxiliary_pretrain_baseline\02_empath...,medium_empathy_synthetic,1,medium_empathy,medium_empathy
5,khách hàng: i fully trust in my intuition and ...,2,data\raw\auxiliary_pretrain_baseline\02_empath...,high_empathy_from_public_empathy_dataset,2,high_empathy,high_empathy
6,khách hàng: i'm about to ride a roller coaster...,1,data\raw\auxiliary_pretrain_baseline\02_empath...,medium_empathy_synthetic,1,medium_empathy,medium_empathy
7,khách hàng: i cant wait for next week! anticip...,2,data\raw\auxiliary_pretrain_baseline\02_empath...,high_empathy_from_public_empathy_dataset,2,high_empathy,high_empathy
8,khách hàng: it was my birthday today! i came h...,1,data\raw\auxiliary_pretrain_baseline\02_empath...,medium_empathy_synthetic,1,medium_empathy,medium_empathy
9,khách hàng: i am worried about an upcoming int...,0,data\raw\auxiliary_pretrain_baseline\02_empath...,low_empathy_synthetic,0,low_empathy,low_empathy


## Dự đoán empathy cho hội thoại mới

In [12]:
def predict_empathy(text, model, tokenizer, id2label):
    model.eval()
    normalized = normalize_text(text)

    inputs = tokenizer(
        normalized,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH,
    )

    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1)[0]
        pred_id = int(torch.argmax(probs).item())

    score_map = {"low_empathy": 2, "medium_empathy": 3, "high_empathy": 5}
    pred_label = id2label[pred_id]

    return {
        "criterion": "empathy",
        "label": pred_label,
        "score": score_map.get(pred_label),
        "confidence": float(probs[pred_id].item()),
        "probabilities": {id2label[i]: float(probs[i].item()) for i in range(len(id2label))},
        "input": text,
    }


examples = [
    """
    Khách hàng: Tôi rất bực vì đơn hàng giao sai sản phẩm, làm mất thời gian của tôi.
    Nhân viên: Dạ em rất xin lỗi anh/chị vì trải nghiệm không tốt này. Em hiểu việc nhận sai sản phẩm rất bất tiện. Anh/chị gửi em mã đơn hàng và hình ảnh sản phẩm, em sẽ kiểm tra và hỗ trợ đổi hàng ngay ạ.
    """,
    """
    Khách hàng: Tôi chờ đơn hàng lâu quá rồi.
    Nhân viên: Gửi mã đơn hàng.
    """,
    """
    Khách hàng: Tôi bị trừ tiền nhưng chưa nhận được đơn.
    Nhân viên: Dạ bên em xin lỗi, anh/chị cho em xin mã giao dịch để em kiểm tra lại tình trạng thanh toán ạ.
    """,
]

for text in examples:
    result = predict_empathy(text, trainer.model, tokenizer, id2label)
    print("\\n" + "-" * 80)
    print(json.dumps(result, ensure_ascii=False, indent=2))

\n--------------------------------------------------------------------------------
{
  "criterion": "empathy",
  "label": "medium_empathy",
  "score": 3,
  "confidence": 1.0,
  "probabilities": {
    "low_empathy": 7.644801458539519e-10,
    "medium_empathy": 1.0,
    "high_empathy": 1.127000270884082e-09
  },
  "input": "\n    Khách hàng: Tôi rất bực vì đơn hàng giao sai sản phẩm, làm mất thời gian của tôi.\n    Nhân viên: Dạ em rất xin lỗi anh/chị vì trải nghiệm không tốt này. Em hiểu việc nhận sai sản phẩm rất bất tiện. Anh/chị gửi em mã đơn hàng và hình ảnh sản phẩm, em sẽ kiểm tra và hỗ trợ đổi hàng ngay ạ.\n    "
}
\n--------------------------------------------------------------------------------
{
  "criterion": "empathy",
  "label": "low_empathy",
  "score": 2,
  "confidence": 1.0,
  "probabilities": {
    "low_empathy": 1.0,
    "medium_empathy": 1.3208829585664716e-09,
    "high_empathy": 1.195097243389398e-09
  },
  "input": "\n    Khách hàng: Tôi chờ đơn hàng lâu quá rồi.\n

## Nhập hội thoại thủ công

In [13]:
while True:
    text = input("\\nNhập hội thoại cần đánh giá empathy, hoặc nhập 'exit' để thoát:\\n> ")

    if text.strip().lower() == "exit":
        break

    result = predict_empathy(text, trainer.model, tokenizer, id2label)
    print(json.dumps(result, ensure_ascii=False, indent=2))

## Ghi chú báo cáo

Cách train này là **baseline bằng pseudo-label** vì dữ liệu empathy public không có nhãn CSKH trực tiếp.

Có thể viết:

```text
Đối với tiêu chí đồng cảm, đề tài sử dụng các bộ dữ liệu hội thoại đồng cảm công khai như EmpatheticDialogues/ESConv để xây dựng mô hình baseline. Do các bộ dữ liệu này không cung cấp trực tiếp nhãn empathy_score theo ngữ cảnh chăm sóc khách hàng tiếng Việt, các phản hồi gốc trong dataset được xem là high_empathy. Đồng thời, đề tài tạo thêm các phản hồi mức low_empathy và medium_empathy theo mẫu hội thoại CSKH để phục vụ huấn luyện ban đầu. Mô hình sau đó cần được fine-tune bằng dữ liệu CSKH tiếng Việt được gán nhãn thủ công để phù hợp hơn với bài toán thực tế.
```